# FASE 1: MEMUAT DATA
Bagian ini berfokus pada persiapan awal lingkungan kerja, pengaturan pustaka yang dibutuhkan, dan pemuatan berbagai dataset (data titik panas, data iklim, peta provinsi, dan data sekolah) ke dalam memori.

In [40]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)
np.random.seed(42)

DATA_RAW = r'd:\Code\Fireline\data\raw'
DATA_PROCESSED = r'd:\Code\Fireline\data\processed'
OUTPUTS = r'd:\Code\Fireline\outputs\figures'

## Memuat Dataset Utama: FIRMS Hotspot Kalimantan
Pada langkah ini, kita memuat dataset utama berupa data titik panas (hotspot) dari instrumen VIIRS satelit NOAA-20.
Langkah ini juga mencakup inspeksi awal seperti pengecekan ukuran data (baris dan kolom), melihat sampel data, 
pengecekan tipe data, serta mengidentifikasi adanya nilai kosong (null) dan duplikasi untuk memahami struktur dasarnya.

In [41]:
print("\n[1.1] Memuat dataset utama FIRMS Hotspot Kalimantan VIIRS NOAA 20")
df_hotspot = pd.read_csv(os.path.join(DATA_RAW, 'Fireline_hotspot_kalimantan_viirs_noaa20_2024_2026.csv'))

print(f"\n[1.2] Ukuran data: {df_hotspot.shape}")
print(f"     Total baris: {df_hotspot.shape[0]:,}")
print(f"     Total kolom: {df_hotspot.shape[1]}")

print("\n[1.3] 5 baris pertama:")
print(df_hotspot.head())
print("\n5 baris terakhir:")
print(df_hotspot.tail())

print("\n[1.4] Tipe data:")
print(df_hotspot.dtypes)

print("\n[1.5] Jumlah nilai kosong per kolom:")
null_counts = df_hotspot.isnull().sum()
print(null_counts)
if null_counts.sum() > 0:
    print(f"\n  Total nilai kosong: {null_counts.sum()}")
    print(f"     Kolom dengan nilai kosong: {list(null_counts[null_counts > 0].index)}")
else:
    print("\n  Tidak ditemukan nilai kosong.")

print("\n[1.6] Pemeriksaan duplikasi:")
exact_dupes = df_hotspot.duplicated().sum()
print(f"     Duplikasi persis: {exact_dupes:,}")

print("\n[1.7] Statistik deskriptif untuk kolom numerik:")
desc = df_hotspot.describe()
print(desc)

print("\n[1.8] Jumlah nilai unik untuk kolom kategorikal:")
for col in ['confidence', 'satellite', 'daynight', 'type', 'instrument', 'version']:
    if col in df_hotspot.columns:
        print(f"\n  {col}:")
        print(f"  {df_hotspot[col].value_counts().to_dict()}")


[1.1] Memuat dataset utama FIRMS Hotspot Kalimantan VIIRS NOAA 20

[1.2] Ukuran data: (61583, 15)
     Total baris: 61,583
     Total kolom: 15

[1.3] 5 baris pertama:
   latitude  longitude  brightness  scan  track    acq_date  acq_time satellite instrument confidence  version  bright_t31    frp daynight  type
0  -2.06932  110.91480      342.89  0.52   0.41  2024-08-01       545       N20      VIIRS          n        2      295.59  32.35        D     0
1  -2.06552  110.91424      335.58  0.52   0.41  2024-08-01       545       N20      VIIRS          n        2      298.05  60.85        D     0
2  -1.53228  114.34785      332.34  0.39   0.36  2024-08-01       545       N20      VIIRS          n        2      296.86   2.77        D     0
3  -0.87858  115.03748      334.21  0.38   0.36  2024-08-01       545       N20      VIIRS          n        2      296.66   2.61        D     0
4  -0.87527  115.03698      342.37  0.38   0.36  2024-08-01       545       N20      VIIRS          n     

## Memuat Dataset Iklim IDN (Climate Data Daily)
Kita memuat data historis iklim harian dan detail stasiun dari BMKG yang akan digunakan nanti untuk korelasi cuaca dan kemunculan api.

In [42]:
print("\n[1.9] Memuat Climate Data Daily IDN")
df_climate = pd.read_csv(os.path.join(DATA_RAW, 'climate_data.csv'))
print(f"     Ukuran: {df_climate.shape}")
print(f"     Kolom: {list(df_climate.columns)}")
print(f"     Data teratas:")
print(df_climate.head(3))

df_station = pd.read_csv(os.path.join(DATA_RAW, 'station_detail.csv'))
print(f"\n     Ukuran detail stasiun: {df_station.shape}")
print(f"     Kolom: {list(df_station.columns)}")
print(df_station.head())


[1.9] Memuat Climate Data Daily IDN
     Ukuran: (589265, 12)
     Kolom: ['date', 'Tn', 'Tx', 'Tavg', 'RH_avg', 'RR', 'ss', 'ff_x', 'ddd_x', 'ff_avg', 'ddd_car', 'station_id']
     Data teratas:
         date    Tn    Tx  Tavg  RH_avg    RR   ss  ff_x  ddd_x  ff_avg ddd_car  station_id
0  01-01-2010  21.4  30.2  27.1    82.0   9.0  0.5   7.0   90.0     5.0      E        96001
1  02-01-2010  21.0  29.6  25.7    95.0  24.0  0.2   6.0   90.0     4.0      E        96001
2  03-01-2010  20.2  26.8  24.5    98.0  63.0  0.0   5.0   90.0     4.0      E        96001

     Ukuran detail stasiun: (192, 7)
     Kolom: ['station_id', 'station_name', 'region_name', 'latitude', 'longitude', 'region_id', 'province_id']
   station_id                                       station_name      region_name  latitude  longitude  region_id  province_id
0       96001                   Stasiun Meteorologi Maimun Saleh      Kota Sabang   5.87655   95.33785         20            1
1       96003  Balai Besar Meteo

## Memuat Dataset Cuaca Pontianak
Data ini merupakan sampel cuaca harian dari satu wilayah spesifik (Pontianak) untuk kepentingan analisis tambahan.

In [43]:
print("\n[1.10] Memuat data Pontianak Weather Daily")
df_pontianak = pd.read_csv(os.path.join(DATA_RAW, 'pontianak_weather_daily_2021_2024.csv'))
print(f"     Ukuran: {df_pontianak.shape}")
print(f"     Kolom: {list(df_pontianak.columns)}")
print(f"     Rentang tanggal: {df_pontianak.iloc[:, 0].min()} hingga {df_pontianak.iloc[:, 0].max()}")
print(df_pontianak.head(3))


[1.10] Memuat data Pontianak Weather Daily
     Ukuran: (1734, 5)
     Kolom: ['date', 'day', 'TAVG', 'RH_AVG', 'RR']
     Rentang tanggal: 01-01-2021 hingga 31-12-2024
         date  day  TAVG  RH_AVG    RR
0  01-01-2021  Fri  26.5    86.0  14.7
1  01-01-2021  Fri  26.5    86.0  14.7
2  02-01-2021  Sat  27.0    86.0   0.5


## Memuat Data Spasial Provinsi Indonesia
Data GeoJSON ini mencakup poligon batas administratif provinsi-provinsi di Indonesia beserta atribut populasi.

In [44]:
print("\n[1.11] Memuat peta provinsi Indonesia beserta populasi")
with open(os.path.join(DATA_RAW, 'indonesia-province-jml-penduduk.json'), 'r') as f:
    province_geojson = json.load(f)

if province_geojson.get('type') == 'FeatureCollection':
    n_features = len(province_geojson.get('features', []))
    print(f"     Tipe: FeatureCollection dengan {n_features} fitur")
    sample_props = province_geojson['features'][0].get('properties', {})
    print(f"     Kunci properti sampel: {list(sample_props.keys())}")
    print(f"     Properti sampel: {sample_props}")
elif province_geojson.get('type') == 'Feature':
    print(f"     Tipe: Fitur Tunggal")
    sample_props = province_geojson.get('properties', {})
    print(f"     Kunci properti: {list(sample_props.keys())}")
else:
    print(f"     Kunci level teratas: {list(province_geojson.keys())[:10]}")
    print(f"     Tipe: {province_geojson.get('type', 'Tidak tersedia')}")


[1.11] Memuat peta provinsi Indonesia beserta populasi
     Tipe: FeatureCollection dengan 32 fitur
     Kunci properti sampel: ['ID', 'kode', 'Propinsi', 'SUMBER', 'Jumlah Penduduk']
     Properti sampel: {'ID': 1, 'kode': 85, 'Propinsi': 'IRIAN JAYA TIMUR', 'SUMBER': 'Peta Dasar BAKOSURTANAL Skala 1 : 250.000', 'Jumlah Penduduk': 1416690}


## Memuat Dataset Infrastruktur (Sekolah)
Data titik lokasi sekolah di Indonesia yang akan membantu kita dalam menentukan skor kerentanan dan paparan sosial terhadap kebakaran.

In [45]:
print("\n[1.12] Memuat dataset sekolah Indonesia")
df_schools = pd.read_csv(os.path.join(DATA_RAW, 'complete_data.csv'))
print(f"     Ukuran: {df_schools.shape}")
print(f"     Kolom: {list(df_schools.columns)}")
has_coords = any(c.lower() in ['latitude', 'longitude', 'lat', 'lon', 'lng'] for c in df_schools.columns)
print(f"     Memiliki kolom koordinat: {has_coords}")
if has_coords:
    print("     Data level titik tersedia untuk penggabungan spasial yang lebih presisi")
else:
    print("     Tidak ada koordinat, hanya data agregat level provinsi")
print(df_schools.head(3))


[1.12] Memuat dataset sekolah Indonesia
     Ukuran: (215371, 11)
     Kolom: ['province_name', 'city_name', 'district_name', 'school_name', 'stage', 'status', 'lat', 'long', 'province_area', 'total_population', 'total_education_age_population']
     Memiliki kolom koordinat: True
     Data level titik tersedia untuk penggabungan spasial yang lebih presisi
  province_name      city_name     district_name        school_name stage status       lat        long  province_area  total_population  total_education_age_population
0         PAPUA  Kab. Tolikara  Kec. Gilungbandu     SMP SATAP KAGI   SMP      N -3.415995  138.339704         319036         4418581.0                         1517814
1         PAPUA  Kab. Tolikara        Kec. Wunin  SD YPPGI WURINERI    SD      S -3.479000  138.680000         319036         4418581.0                         1517814
2         PAPUA  Kab. Tolikara    Kec. Bokondini     SMAN BOKONDINI   SMA      N -3.531731  138.658234         319036         4418581.0 

## Ringkasan Inventaris Data
Seluruh dataset utama dan pendukung telah dimuat. Berikut adalah ringkasan jumlah baris atau elemen dari setiap dataset.

In [46]:
print("\n[1.15] RINGKASAN INVENTARIS DATA")
print(f"FIRMS Hotspot Kalimantan: {df_hotspot.shape[0]:,} baris")
print(f"Climate Data Daily IDN: {df_climate.shape[0]:,} baris")
print(f"Detail Stasiun: {df_station.shape[0]:,} baris")
print(f"Cuaca Pontianak Harian: {df_pontianak.shape[0]:,} baris")
print(f"Peta Provinsi dan Populasi: {n_features if 'n_features' in dir() else '?'} fitur")
print(f"Dataset Sekolah: {df_schools.shape[0]:,} baris")


[1.15] RINGKASAN INVENTARIS DATA
FIRMS Hotspot Kalimantan: 61,583 baris
Climate Data Daily IDN: 589,265 baris
Detail Stasiun: 192 baris
Cuaca Pontianak Harian: 1,734 baris
Peta Provinsi dan Populasi: 32 fitur
Dataset Sekolah: 215,371 baris


# FASE 2: AUDIT KUALITAS DATA
Pada fase ini, kita akan memeriksa kualitas data, memverifikasi kesesuaian struktur, dan membersihkan anomali pada dataset utama.

## Konfirmasi Skema dan Penanganan Waktu
Kita mencocokkan ketersediaan kolom sesuai dokumen FIRMS VIIRS dan menyesuaikan format zona waktu (UTC ke WIB) agar konsisten dengan lokasi Kalimantan.

In [47]:
print("\n[2.1] Konfirmasi skema data berdasarkan dokumentasi FIRMS VIIRS")
expected_columns = ['latitude', 'longitude', 'brightness', 'scan', 'track', 
                    'acq_date', 'acq_time', 'satellite', 'instrument', 
                    'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type']
actual_columns = list(df_hotspot.columns)
print(f"     Kolom yang diharapkan: {expected_columns}")
print(f"     Kolom aktual: {actual_columns}")

for exp in expected_columns:
    if exp not in actual_columns:
        if exp == 'brightness' and 'bright_ti4' in actual_columns:
            print(f"     Kolom {exp} tidak ditemukan, tetapi ada bright_ti4")
        elif exp == 'bright_t31' and 'bright_ti5' in actual_columns:
            print(f"     Kolom {exp} tidak ditemukan, tetapi ada bright_ti5")
        else:
            print(f"     Kolom hilang: {exp}")

if 'brightness' in actual_columns:
    print("     Kolom bernama brightness ditemukan, interpretasi sebagai temperatur kecerahan dalam Kelvin.")

print("\n[2.2] Penanganan waktu konversi ke WIB")
df_hotspot['acq_date'] = pd.to_datetime(df_hotspot['acq_date'])

df_hotspot['acq_time_str'] = df_hotspot['acq_time'].astype(str).str.zfill(4)
df_hotspot['hour_utc'] = df_hotspot['acq_time_str'].str[:2].astype(int)
df_hotspot['minute_utc'] = df_hotspot['acq_time_str'].str[2:].astype(int)

df_hotspot['datetime_utc'] = pd.to_datetime(
    df_hotspot['acq_date'].dt.strftime('%Y-%m-%d') + ' ' + 
    df_hotspot['acq_time_str'].str[:2] + ':' + df_hotspot['acq_time_str'].str[2:],
    format='%Y-%m-%d %H:%M'
)

df_hotspot['datetime_wib'] = df_hotspot['datetime_utc'] + pd.Timedelta(hours=7)
df_hotspot['hour_local'] = df_hotspot['datetime_wib'].dt.hour
df_hotspot['date_local'] = df_hotspot['datetime_wib'].dt.date

print(f"     Rentang tanggal UTC: {df_hotspot['acq_date'].min()} hingga {df_hotspot['acq_date'].max()}")
print(f"     Rentang tanggal WIB: {df_hotspot['datetime_wib'].min()} hingga {df_hotspot['datetime_wib'].max()}")


[2.1] Konfirmasi skema data berdasarkan dokumentasi FIRMS VIIRS
     Kolom yang diharapkan: ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type']
     Kolom aktual: ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type']
     Kolom bernama brightness ditemukan, interpretasi sebagai temperatur kecerahan dalam Kelvin.

[2.2] Penanganan waktu konversi ke WIB
     Rentang tanggal UTC: 2024-08-01 00:00:00 hingga 2026-05-31 00:00:00
     Rentang tanggal WIB: 2024-08-01 12:45:00 hingga 2026-06-01 01:06:00


## Pembersihan Data: Duplikasi, Penyaringan Tipe, dan Kepercayaan
Bagian ini menghapus data duplikat persis, menyaring titik api hanya untuk kebakaran vegetasi (type 0), dan menyandikan nilai kepercayaan (confidence) menjadi angka terurut.

In [48]:
print("\n[2.3] Pemeriksaan duplikasi dan tumpang tindih")
exact_dupes = df_hotspot.duplicated().sum()
print(f"     Baris duplikat persis: {exact_dupes:,}")

key_cols = ['latitude', 'longitude', 'acq_date', 'acq_time']
near_dupes = df_hotspot.duplicated(subset=key_cols, keep=False).sum()
print(f"     Duplikat mirip lokasi dan waktu yang sama: {near_dupes:,}")

if exact_dupes > 0:
    print(f"     Menghapus {exact_dupes} duplikat persis")
    df_hotspot = df_hotspot.drop_duplicates().reset_index(drop=True)

print("\n[2.4] Penyaringan tipe data")
type_dist = df_hotspot['type'].value_counts()
print(f"     Distribusi tipe:")
for t, c in type_dist.items():
    label = {0: 'kebakaran vegetasi', 1: 'gunung berapi', 
             2: 'sumber darat statis lainnya', 3: 'lepas pantai'}.get(t, 'tidak diketahui')
    print(f"       tipe {t} {label}: {c:,} ({c/len(df_hotspot)*100:.1f} persen)")

non_veg_count = len(df_hotspot[df_hotspot['type'] != 0])
if non_veg_count > 0:
    print(f"     Mempertahankan tipe 0 saja dan menghapus {non_veg_count:,} deteksi non vegetasi")
    df_fire = df_hotspot[df_hotspot['type'] == 0].copy()
else:
    df_fire = df_hotspot.copy()

print("\n[2.5] Penyandian tingkat kepercayaan")
confidence_map = {'l': 1, 'n': 2, 'h': 3}
if df_fire['confidence'].dtype == object:
    df_fire['confidence_ord'] = df_fire['confidence'].map(confidence_map)
    print(f"     Tingkat kepercayaan berhasil dipetakan ke angka 1 2 3")


[2.3] Pemeriksaan duplikasi dan tumpang tindih
     Baris duplikat persis: 0
     Duplikat mirip lokasi dan waktu yang sama: 0

[2.4] Penyaringan tipe data
     Distribusi tipe:
       tipe 0 kebakaran vegetasi: 61,583 (100.0 persen)

[2.5] Penyandian tingkat kepercayaan


## Kewajaran Geografis dan Waktu
Memastikan semua titik api benar-benar berada di dalam kotak koordinat (*bounding box*) Kalimantan dan memeriksa kelengkapan data waktu tahunan.

In [49]:
print("\n[2.6] Pemeriksaan kewajaran geografis")
lat_min, lat_max = df_fire['latitude'].min(), df_fire['latitude'].max()
lon_min, lon_max = df_fire['longitude'].min(), df_fire['longitude'].max()
KALIM_LAT_MIN, KALIM_LAT_MAX = -4.5, 3.5
KALIM_LON_MIN, KALIM_LON_MAX = 108.5, 117.5

outside = df_fire[
    ~((df_fire['latitude'].between(KALIM_LAT_MIN, KALIM_LAT_MAX)) & 
      (df_fire['longitude'].between(KALIM_LON_MIN, KALIM_LON_MAX)))
]

if len(outside) > 0:
    print(f"     Ada {len(outside):,} titik di luar area Kalimantan")
    df_fire = df_fire[
        (df_fire['latitude'].between(KALIM_LAT_MIN, KALIM_LAT_MAX)) & 
        (df_fire['longitude'].between(KALIM_LON_MIN, KALIM_LON_MAX))
    ].reset_index(drop=True)
else:
    print(f"     Semua titik berada di dalam area Kalimantan")

print("\n[2.7] Pemeriksaan kelengkapan waktu")
df_fire['year'] = df_fire['acq_date'].dt.year
df_fire['month'] = df_fire['acq_date'].dt.month

year_counts = df_fire.groupby('year').size()
for y, c in year_counts.items():
    print(f"       Tahun {y}: {c:,} titik panas")


[2.6] Pemeriksaan kewajaran geografis
     Ada 3,798 titik di luar area Kalimantan

[2.7] Pemeriksaan kelengkapan waktu
       Tahun 2024: 23,050 titik panas
       Tahun 2025: 28,394 titik panas
       Tahun 2026: 6,341 titik panas


## Analisis Anomali Intensitas Kebakaran dan Area Piksel
Memeriksa distribusi nilai Fire Radiative Power (FRP) untuk menemukan kejadian *mega-fire* ekstrem serta menghitung perkiraan luas area yang ditangkap setiap piksel berdasarkan atribut pindaian satelit.

In [50]:
print("\n[2.8] Analisis anomali FRP")
frp_desc = df_fire['frp'].describe(percentiles=[.25, .5, .75, .9, .95, .99])
print(frp_desc)
print(f"\n     Rentang FRP: {df_fire['frp'].min():.2f} hingga {df_fire['frp'].max():.2f} MW")

top_01 = df_fire['frp'].quantile(0.999)
mega_fires = df_fire[df_fire['frp'] >= top_01]
print(f"     Ambang batas persentil 99.9: {top_01:.2f} MW")
print(f"     Kejadian ekstrem di atas ambang batas: {len(mega_fires):,} titik")

print("\n[2.9] Perhitungan area piksel")
df_fire['pixel_area_km2'] = df_fire['scan'] * df_fire['track']
print(f"     Rata rata area piksel: {df_fire['pixel_area_km2'].mean():.4f} kilometer persegi")


[2.8] Analisis anomali FRP
count    57785.000000
mean        10.812202
std         17.799874
min          0.090000
25%          3.650000
50%          6.270000
75%         11.730000
90%         22.620000
95%         33.628000
99%         71.511600
max        954.790000
Name: frp, dtype: float64

     Rentang FRP: 0.09 hingga 954.79 MW
     Ambang batas persentil 99.9: 204.36 MW
     Kejadian ekstrem di atas ambang batas: 60 titik

[2.9] Perhitungan area piksel
     Rata rata area piksel: 0.2273 kilometer persegi


## Validasi Keselarasan Data Lintas Kumpulan Data
Mengkonfirmasi apakah tanggal pada dataset iklim cocok dengan data titik panas, serta memvalidasi ketersediaan batas provinsi Kalimantan.

In [51]:
print("\n[2.10] Pengecekan keselarasan rentang tanggal dataset iklim")
if 'date' in df_climate.columns:
    climate_dates = pd.to_datetime(df_climate['date'], dayfirst=True, errors='coerce')
elif df_climate.columns[0] not in ['Tn', 'Tx', 'Tavg']:
    climate_dates = pd.to_datetime(df_climate.iloc[:, 0], errors='coerce')
else:
    climate_dates = None

if climate_dates is not None and not climate_dates.isna().all():
    print(f"     Rentang data iklim dari {climate_dates.min().strftime('%Y-%m-%d')} hingga {climate_dates.max().strftime('%Y-%m-%d')}")

print("\n[2.11] Validasi batas provinsi")
if province_geojson.get('type') == 'FeatureCollection':
    features = province_geojson['features']
    province_names = []
    for f in features:
        props = f.get('properties', {})
        name = props.get('name', props.get('NAME_1', props.get('Provinsi', props.get('province', 'tidak diketahui'))))
        province_names.append(name)
    
    kalimantan_provinces = [p for p in province_names if p and ('kalimantan' in str(p).lower() or 'kalim' in str(p).lower())]
    print(f"     Provinsi Kalimantan yang ditemukan: {kalimantan_provinces}")


[2.10] Pengecekan keselarasan rentang tanggal dataset iklim
     Rentang data iklim dari 2010-01-01 hingga 2020-12-31

[2.11] Validasi batas provinsi
     Provinsi Kalimantan yang ditemukan: []


## Penyimpanan Data Akhir Fase 1 & 2
Menambahkan atribut penanggalan yang diekstrak (tahun, bulan, hari) untuk mempermudah analisis temporal selanjutnya, kemudian menyimpan data yang sudah bersih ke dalam format CSV.

In [52]:
df_fire['day_of_week'] = df_fire['datetime_wib'].dt.day_name()
df_fire['week'] = df_fire['datetime_wib'].dt.isocalendar().week.astype(int)

output_path = os.path.join(DATA_PROCESSED, 'hotspot_cleaned.csv')
df_fire.to_csv(output_path, index=False)
print(f"\nData titik panas yang telah dibersihkan berhasil disimpan ke: {output_path}")
print("FASE 1 DAN 2 SELESAI")


Data titik panas yang telah dibersihkan berhasil disimpan ke: d:\Code\Fireline\data\processed\hotspot_cleaned.csv
FASE 1 DAN 2 SELESAI
